# SignBridge ML Evaluation Visualizations
This notebook provides industry-standard Machine Learning visualizations for evaluating the SignBridge Custom MLP model against the Baseline model.

**Note:** Dynamic versions of these 7 core charts (alongside Radar capability profiles and Confidence Distributions) are already implemented **within the application code** itself via `src/components/ResearchDashboard.tsx`. To view them in the browser, run `npm run dev` and navigate to the **Research Dashboard** tab.

This notebook uses Python libraries (`matplotlib`, `seaborn`, `scikit-learn`) to generate static publication-quality figures from synthetic evaluation outputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context('notebook', font_scale=1.2)

gestures = ['Hello', 'Thank You', 'Yes', 'No', 'Please', 'Sorry', 'Help', 'A', 'B', 'C']
n_classes = len(gestures)

## 1. Learning Curves (Accuracy & Loss)
Shows the model's convergence over epochs on training and validation sets. This diagram is crucial for identifying overfitting or underfitting (bias/variance tradeoff).

In [ ]:
epochs = np.arange(1, 51)

# Synthetic cross-entropy loss
train_loss = 2.5 * np.exp(-epochs/10) + np.random.normal(0, 0.05, 50)
val_loss = 2.5 * np.exp(-epochs/9) + 0.2 + np.random.normal(0, 0.08, 50)
val_loss[35:] += np.linspace(0, 0.3, 15)  # Simulate Early Stopping trigger

# Synthetic accuracy
train_acc = 0.2 + 0.75 * (1 - np.exp(-epochs/8)) + np.random.normal(0, 0.01, 50)
val_acc = 0.2 + 0.7 * (1 - np.exp(-epochs/7)) + np.random.normal(0, 0.02, 50)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Loss Plot
ax1.plot(epochs, train_loss, 'b-', label='Training Loss', linewidth=2.5)
ax1.plot(epochs, val_loss, 'r--', label='Validation Loss', linewidth=2.5)
ax1.axvline(x=35, color='gray', linestyle=':', label='Early Stopping Point')
ax1.set_title('Model Loss (Categorical Cross-Entropy)', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

# Accuracy Plot
ax2.plot(epochs, train_acc, 'b-', label='Training Accuracy', linewidth=2.5)
ax2.plot(epochs, val_acc, 'r--', label='Validation Accuracy', linewidth=2.5)
ax2.axvline(x=35, color='gray', linestyle=':', label='Optimal Convergence')
ax2.axhline(y=0.9, color='g', linestyle='-.', label='Target 90% Acc')
ax2.set_title('Model Accuracy Convergence', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

## 2. Confusion Matrix Heatmap
Visualizes the classification performance across all gesture categories, highlighting True Positives along the diagonal, and False Positives/Negatives everywhere else.

In [ ]:
# Generate synthetic confusion matrix with strong diagonal
cm = np.zeros((n_classes, n_classes), dtype=int)
for i in range(n_classes):
    for j in range(n_classes):
        if i == j:
            cm[i, j] = np.random.randint(45, 65)
        else:
            cm[i, j] = np.random.randint(0, 4)

# Add some specific common inter-class confusions
cm[gestures.index('A'), gestures.index('Help')] += 8
cm[gestures.index('No'), gestures.index('Yes')] += 5

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', 
            xticklabels=gestures, yticklabels=gestures,
            linewidths=.5, cbar_kws={'label': 'Number of Predictions'})
plt.title('Custom MLP Confusion Matrix', pad=20, fontweight='bold', fontsize=16)
plt.xlabel('Predicted Gesture', labelpad=15, fontweight='bold')
plt.ylabel('True Gesture', labelpad=15, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Precision-Recall Curve Comparison
Demonstrates the tradeoff between precision and recall for different threshold values, which is preferred over ROC curves for imbalanced multi-class classification setups.

In [ ]:
recall = np.linspace(0, 1, 100)
# Synthetic PR curve equation for visual fidelity
precision_custom = np.clip(1.0 - 0.1 * recall**4 - 0.03 * np.random.rand(100), 0, 1)
precision_baseline = np.clip(0.95 - 0.4 * recall**3 - 0.05 * np.random.rand(100), 0, 1)

plt.figure(figsize=(10, 8))
plt.plot(recall, precision_custom, color='#f97316', label='Custom MLP (mAP = 0.95)', linewidth=3)
plt.plot(recall, precision_baseline, color='#3b82f6', linestyle='--', label='Baseline Model (mAP = 0.81)', linewidth=3)

plt.fill_between(recall, precision_custom, alpha=0.1, color='#f97316')
plt.fill_between(recall, precision_baseline, alpha=0.1, color='#3b82f6')

plt.title('Precision-Recall Curve Comparison', fontweight='bold', fontsize=16)
plt.xlabel('Recall (Sensitivity)', fontweight='bold')
plt.ylabel('Precision (Positive Predictive Value)', fontweight='bold')
plt.xlim([0.0, 1.0])
plt.ylim([0.4, 1.05])
plt.legend(loc='lower left', frameon=True, shadow=True)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 4. Inference Latency Distribution (Box Plot)
Shows the execution time comparison and consistency. Variability in inference time must be low for a smooth 30 FPS real-time user experience.

In [ ]:
# Synthetic latency data (ms)
baseline_latency = np.random.lognormal(mean=3.4, sigma=0.2, size=500)
custom_latency = np.random.normal(loc=14.5, scale=2.5, size=500)
custom_latency = np.clip(custom_latency, 5, None)

data = pd.DataFrame({
    'Latency (ms)': np.concatenate([baseline_latency, custom_latency]),
    'Model': ['Baseline Model']*500 + ['Custom MLP (In-Browser)']*500
})

plt.figure(figsize=(10, 6))
sns.boxplot(x='Latency (ms)', y='Model', data=data, palette=['#3b82f6', '#f97316'], showfliers=True,
            boxprops={'alpha': 0.8})

plt.axvline(x=33.3, color='#ef4444', linestyle='--', linewidth=2.5, label='Real-time Threshold (33ms / 30 FPS)')

plt.title('Inference Latency Distribution (Client-Side Web Environment)', fontweight='bold')
plt.xlabel('Latency per Inference (milliseconds)', fontweight='bold')
plt.ylabel('Model Architecture', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()